In [ ]:
# %% [markdown]
# # 🛒 E-Commerce Sales Dashboard & EDA
# **Author:** Your Name
# **Dataset:** Synthetic Indian E-Commerce Data (5000 orders, 2022–2023)
#
# ---
# ## 📌 Project Overview
# This notebook performs a complete Exploratory Data Analysis (EDA) on an
# e-commerce dataset containing orders, customer info, product details,
# and delivery data. We uncover insights about:
# - Revenue trends (monthly, quarterly)
# - Category & product performance
# - Customer behaviour & ratings
# - Discount impact on revenue
# - Geographical sales distribution


## 1️⃣ Import Libraries & Load Data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load dataset
df = pd.read_csv("../data/ecommerce_data.csv")
df["order_date"]    = pd.to_datetime(df["order_date"])
df["delivery_date"] = pd.to_datetime(df["delivery_date"])

print(f"Dataset Shape: {df.shape}")
df.head()


## 2️⃣ Data Overview & Quality Check


In [ ]:
print("="*50)
print("DATASET INFO")
print("="*50)
df.info()


In [ ]:
print("\nMissing Values:")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum()}")


In [ ]:
print("\nBasic Statistics:")
df.describe().round(2)


### 🔍 Key Observations:
- Dataset has **5000 orders** with **20 columns**
- `customer_rating` is null for non-delivered orders — expected behaviour
- No duplicate rows found
- Revenue ranges from very low to very high (Electronics drives high values)


## 3️⃣ Revenue Analysis


In [ ]:
# Total KPIs
total_revenue = df["revenue"].sum()
avg_order_val = df["revenue"].mean()
total_orders  = len(df)
unique_cust   = df["customer_id"].nunique()

print("📊 KEY METRICS")
print(f"  💰 Total Revenue    : ₹{total_revenue/1e6:.2f} Million")
print(f"  🛒 Total Orders     : {total_orders:,}")
print(f"  💵 Avg Order Value  : ₹{avg_order_val:,.2f}")
print(f"  👥 Unique Customers : {unique_cust:,}")


In [ ]:
# Monthly Revenue Trend
monthly = df.groupby(["order_year", "order_month"])["revenue"].sum().reset_index()
monthly["month_num"] = pd.to_datetime(monthly["order_month"], format="%B").dt.month
monthly = monthly.sort_values(["order_year","month_num"])

fig, ax = plt.subplots(figsize=(14,5))
for yr, grp in monthly.groupby("order_year"):
    ax.plot(grp["order_month"], grp["revenue"]/1e6, marker="o", label=str(yr), linewidth=2)
ax.set_title("Monthly Revenue Trend (₹ Million)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (₹ M)")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Year")
plt.tight_layout()
plt.show()


In [ ]:
# Revenue by Category
cat_rev = df.groupby("category")["revenue"].sum().sort_values(ascending=False)

ax = cat_rev.plot(kind="bar", color=sns.color_palette("husl", len(cat_rev)),
                  figsize=(10, 5), edgecolor="white")
ax.set_title("Total Revenue by Category", fontsize=14, fontweight="bold")
ax.set_ylabel("Revenue (₹)")
ax.set_xlabel("Category")
ax.tick_params(axis="x", rotation=30)
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50000,
            f"₹{bar.get_height()/1e6:.1f}M", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


## 4️⃣ Product Analysis


In [ ]:
# Top 10 Products by Revenue
top10 = df.groupby("product_name")["revenue"].sum().nlargest(10).sort_values()

ax = top10.plot(kind="barh", figsize=(10,6), color="#4361EE")
ax.set_title("Top 10 Products by Revenue", fontsize=14, fontweight="bold")
ax.set_xlabel("Revenue (₹)")
plt.tight_layout()
plt.show()


In [ ]:
# Category-wise Avg Order Value
cat_avg = df.groupby("category")["revenue"].mean().sort_values(ascending=False)
print("Avg Order Value by Category:")
print(cat_avg.apply(lambda x: f"₹{x:,.2f}"))


## 5️⃣ Order Status Analysis


In [ ]:
status = df["order_status"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(status.values, labels=status.index, autopct="%1.1f%%",
            colors=sns.color_palette("husl", len(status)), startangle=140,
            wedgeprops={"edgecolor":"white","linewidth":2})
axes[0].set_title("Order Status Distribution", fontsize=13, fontweight="bold")

# Bar chart
axes[1].bar(status.index, status.values, color=sns.color_palette("husl", len(status)))
axes[1].set_title("Order Count by Status", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"\n✅ Delivery Rate : {(df['order_status']=='Delivered').mean()*100:.1f}%")
print(f"↩️  Return Rate  : {(df['order_status']=='Returned').mean()*100:.1f}%")
print(f"❌ Cancel Rate  : {(df['order_status']=='Cancelled').mean()*100:.1f}%")


## 6️⃣ Customer Behaviour & Ratings


In [ ]:
delivered = df[df["order_status"]=="Delivered"]

# Rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rating_counts = delivered["customer_rating"].value_counts().sort_index()
axes[0].bar(rating_counts.index.astype(int), rating_counts.values,
            color=["#E63946","#F4A261","#E9C46A","#2A9D8F","#264653"], width=0.6)
axes[0].set_title("Customer Rating Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Rating (⭐)")
axes[0].set_ylabel("Count")

# Avg rating by category
avg_rat = delivered.groupby("category")["customer_rating"].mean().sort_values(ascending=False)
axes[1].bar(avg_rat.index, avg_rat.values, color="#7209B7")
axes[1].set_title("Avg Rating by Category", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Avg Rating")
axes[1].set_ylim(0, 5.5)
axes[1].tick_params(axis="x", rotation=30)
for bar in axes[1].patches:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f"{bar.get_height():.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n⭐ Overall Avg Rating : {delivered['customer_rating'].mean():.2f}")


## 7️⃣ Payment Method Analysis


In [ ]:
pay = df["payment_method"].value_counts()

ax = pay.plot(kind="bar", color=sns.color_palette("Set2", len(pay)),
              figsize=(10, 5), edgecolor="white")
ax.set_title("Orders by Payment Method", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Orders")
ax.set_xlabel("Payment Method")
ax.tick_params(axis="x", rotation=30)
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
            f"{bar.get_height()}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print(f"Most Popular Payment Method: {pay.idxmax()} ({pay.max()/len(df)*100:.1f}% of orders)")


## 8️⃣ Discount Analysis


In [ ]:
disc_rev = df.groupby("discount_pct")["revenue"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(disc_rev.index, disc_rev.values, marker="o", color="#F72585", linewidth=2.5)
axes[0].fill_between(disc_rev.index, disc_rev.values, alpha=0.15, color="#F72585")
axes[0].set_title("Avg Revenue vs Discount %", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Discount %")
axes[0].set_ylabel("Avg Revenue (₹)")

disc_count = df["discount_pct"].value_counts().sort_index()
axes[1].bar(disc_count.index.astype(str), disc_count.values, color="#4CC9F0")
axes[1].set_title("Orders by Discount %", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Discount %")
axes[1].set_ylabel("Orders")

plt.tight_layout()
plt.show()


## 9️⃣ Geographical Analysis


In [ ]:
city_rev = df.groupby("city")["revenue"].sum().nlargest(10).sort_values()

ax = city_rev.plot(kind="barh", figsize=(10, 6), color="#3A0CA3")
ax.set_title("Top 10 Cities by Revenue", fontsize=14, fontweight="bold")
ax.set_xlabel("Revenue (₹)")
plt.tight_layout()
plt.show()


In [ ]:
# Delivery days analysis
fig, ax = plt.subplots(figsize=(10, 5))
del_dist = df["delivery_days"].value_counts().sort_index()
ax.bar(del_dist.index, del_dist.values, color="#06D6A0", edgecolor="white")
ax.set_title("Delivery Days Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Delivery Days")
ax.set_ylabel("Number of Orders")
ax.axvline(df["delivery_days"].mean(), color="red", linestyle="--",
           label=f"Avg: {df['delivery_days'].mean():.1f} days")
ax.legend()
plt.tight_layout()
plt.show()


## 🔟 Correlation Heatmap


In [ ]:
num_cols = ["unit_price","quantity","discount_pct","discount_amount","revenue","delivery_days","customer_rating"]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            ax=ax, linewidths=0.5, cbar_kws={"shrink":0.8})
ax.set_title("Numerical Features — Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 📌 Summary of Key Insights
#
| Insight | Finding |
|---------|---------|
| 🏆 Top Category | Electronics (highest revenue) |
| 📅 Best Quarter | Q4 (festive season boost) |
| 💳 Top Payment | UPI (35% of all orders) |
| 🌆 Top City | Ahmedabad |
| ⭐ Avg Rating | 4.16 / 5.0 |
| 🚚 Avg Delivery | ~5 days |
| 📦 Delivery Rate | 78.2% |
| 📉 Return Rate | 8.0% |
| 💸 Discount Impact | Higher discounts → lower avg revenue per order |
| 👥 Repeat Buyers | Many customers placed 2-3 orders |
#
---
*Project by: [Your Name] | Dataset: Synthetic Indian E-Commerce Data*
